In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sharankumare/mcq-test/test.csv
/kaggle/input/datasets/sharankumare/mcq-train/train.csv


In [2]:
from datasets import load_dataset

# Load train.csv
Train = load_dataset(
    "csv",
    data_files="/kaggle/input/datasets/sharankumare/mcq-train/train.csv"
)["train"]

# Create combined_text = prompt + " " + A
Train = Train.map(
    lambda x: {
        "combined_text": x["prompt"] + " " + x["A"]
    }
)

# Character length of row 51
print(len(Train[51]["combined_text"]))

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

614


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer.sep_token)
print(tokenizer.sep_token_id)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[SEP]
102


In [4]:
print(Train)
print(type(Train))

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'combined_text'],
    num_rows: 2000
})
<class 'datasets.arrow_dataset.Dataset'>


In [5]:
print(type(Train["prompt"]))
print(type(Train["prompt"][0]))
print(Train["prompt"][0])

<class 'datasets.arrow_dataset.Column'>
<class 'str'>
Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.


In [6]:
encodings = tokenizer(
    list(Train["prompt"]),
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(encodings["input_ids"].shape)

torch.Size([2000, 128])


In [7]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("bert-base-uncased")

hidden_size = config.hidden_size
num_heads = config.num_attention_heads

head_dim = hidden_size // num_heads

print("Hidden Size:", hidden_size)
print("Attention Heads:", num_heads)
print("Head Dimension:", head_dim)

Hidden Size: 768
Attention Heads: 12
Head Dimension: 64


In [8]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# Tokenize row 0 prompt (default settings)
inputs = tokenizer(Train[0]["prompt"], return_tensors="pt")

# Forward pass
outputs = model(**inputs)

# Print shape
print(outputs.last_hidden_state.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])


In [9]:
# Extract CLS embedding (batch 0, token 0)
cls_embedding = outputs.last_hidden_state[0, 0]

# Sum first 5 values
answer = cls_embedding[:5].sum().item()

print(round(answer, 4))

-1.2001


In [10]:
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

inputs = tokenizer("Light-ion fusion is a technique.", return_tensors="pt")
outputs = model(**inputs)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
fusion_index = tokens.index("fusion")

attention = outputs.attentions[-1][0, 0]

print(round(attention[0, fusion_index].item(), 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.1025


In [11]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Get prompt and Option B from row 0
prompt = Train[0]["prompt"]
option_b = Train[0]["B"]

# Generate embeddings
prompt_embedding = model.encode(prompt, convert_to_tensor=True)
option_b_embedding = model.encode(option_b, convert_to_tensor=True)

# Cosine similarity
similarity = util.cos_sim(prompt_embedding, option_b_embedding)

print(round(similarity.item(), 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.7658


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

mini_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

options = ["A", "B", "C", "D", "E"]

mini_map = 0
improved = 0

for row in Train:

    prompt = row["prompt"]
    answer = row["answer"]

    # Pipeline 1 : TF-IDF

    docs = [prompt] + [row[o] for o in options]

    tfidf = TfidfVectorizer()
    X = tfidf.fit_transform(docs)

    sims = cosine_similarity(X[0], X[1:]).flatten()

    tfidf_rank = [
        options[i]
        for i in sims.argsort()[::-1]
    ]


    # Pipeline 2 : MiniLM

    prompt_emb = mini_model.encode(prompt, convert_to_tensor=True)
    option_embs = mini_model.encode(
        [row[o] for o in options],
        convert_to_tensor=True
    )

    sims = util.cos_sim(prompt_emb, option_embs)[0].cpu().numpy()

    mini_rank = [
        options[i]
        for i in sims.argsort()[::-1]
    ]

    # MAP@3
    top3 = mini_rank[:3]

    if answer in top3:
        mini_map += 1 / (top3.index(answer) + 1)


    # Improvement count
    tfidf_hit = answer in tfidf_rank[:3]
    mini_hit = answer in mini_rank[:3]

    if (not tfidf_hit) and mini_hit:
        improved += 1

mini_map /= len(Train)

print("MiniLM MAP@3:", round(mini_map, 4))
print("Improved count:", improved)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MiniLM MAP@3: 0.4231
Improved count: 564


In [13]:
from transformers import pipeline
# Initialize zero-shot classifier
classifier = pipeline("zero-shot-classification")

# Prompt from row index 1
prompt = Train[1]["prompt"]

# Candidate labels: Options A, B, C
candidate_labels = [
    Train[1]["A"],
    Train[1]["B"],
    Train[1]["C"]
]

# Predict
result = classifier(
    prompt,
    candidate_labels=candidate_labels
)

# Probability score of the top-ranked option
print(round(result["scores"][0], 4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

0.4575


In [14]:
result_softmax = classifier(
    prompt,
    candidate_labels=candidate_labels
)

# This question (Independent Sigmoids)
result_sigmoid = classifier(
    prompt,
    candidate_labels=candidate_labels,
    multi_label=True
)

softmax_sum = sum(result_softmax["scores"])
sigmoid_sum = sum(result_sigmoid["scores"])

print("Softmax sum:", softmax_sum)
print("Sigmoid sum:", sigmoid_sum)
print("Absolute difference:", round(abs(softmax_sum - sigmoid_sum), 4))

Softmax sum: 0.9999999701976776
Sigmoid sum: 0.0005096072964079212
Absolute difference: 0.9995


In [15]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

text = (
    f"Question: {Train[0]['prompt']}. "
    f"Is the correct answer A: {Train[0]['A']} "
    f"or B: {Train[0]['B']}? "
    f"Answer with just the letter A or B."
)

inputs = tokenizer(text, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=5
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

B
